# 00 — Environment and data profile (rel-f1)

**Purpose.** Before anything is trained, confirm that the data loads on the uniform
anchor grid the rerun uses, and record what each update and window contains:
rows, entities, held-out entities. Also measure seconds per training step, the
unit every later cost estimate is built on.

**Gate.**
1. Every task loads, and its plan has update times and windows.
2. Tasks with at least 5 updates are main-grid tasks; fewer go to the appendix.
3. Every window with rows has at least one held-out entity.
4. The step-time benchmark is finite and saved.

In [1]:
DATASET = "rel-f1"
TASKS = ["driver-position", "driver-dnf", "driver-top3"]
DEVICE = "auto"
SCALE = "smoke"  # "smoke" (CPU, minutes) or "full"
OUT = "notebooks/methodology/out"
THREADS = 8
HOLDOUT_FRAC = 0.1

In [2]:
# Parameters
SCALE = "smoke"
DEVICE = "cpu"


In [3]:
import json
import os
import platform
import sys
import time
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import torch

torch.set_num_threads(THREADS)
out = Path(OUT)
out.mkdir(parents=True, exist_ok=True)

from experiments.continuous_learning.chain_runner import ChainConfig, build_context
from redelex.continual.anchors import native_anchor_timestamps

/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## Environment

In [4]:
from importlib.metadata import version

env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "relbench": version("relbench"),
    "torch_geometric": version("torch_geometric"),
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor() or "cpu",
    "threads": torch.get_num_threads(),
    "host": platform.node(),
}
pd.Series(env)

/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


python                  3.12.3
torch              2.9.1+cu128
relbench                 2.1.1
torch_geometric          2.7.0
cuda_available           False
device_name             x86_64
threads                      8
host               dev-sandbox
dtype: object

## Load every task on the uniform grid

Anchors are placed every Δw through the RelBench validation timestamp. Updates are
placed every native ΔI (the RelBench test − validation span), starting from the
first update that has at least `min_initial_rows` trainable rows.

In [5]:
contexts = {}
load_seconds = {}
for task_name in TASKS:
    cfg = ChainConfig(DATASET, task_name, "scratch_all", 0, OUT, device=DEVICE, holdout_frac=HOLDOUT_FRAC)
    t0 = time.perf_counter()
    contexts[task_name] = build_context(cfg)
    load_seconds[task_name] = time.perf_counter() - t0

summary = []
for task_name, ctx in contexts.items():
    p, plan = ctx.protocol, ctx.plan
    summary.append(
        {
            "task": task_name,
            "type": ctx.task.task_type.value,
            "Δw (days)": p.delta_w.days,
            "ΔI (days)": p.delta_i.days,
            "anchors": ctx.rows["time"].nunique(),
            "rows": len(ctx.rows),
            "entities": ctx.rows["entity"].nunique(),
            "held-out entities": ctx.rows.loc[ctx.rows.holdout, "entity"].nunique(),
            "updates": plan.num_updates,
            "first update": plan.update_times[0].date(),
            "last update": plan.update_times[-1].date(),
            "load s": round(load_seconds[task_name], 1),
        }
    )
summary = pd.DataFrame(summary).set_index("task")
summary

Loading Database object from /home/pelesjak/.cache/relbench/rel-f1/db...
Done in 0.08 seconds.


/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.se

/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.se

/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.se

,type,Δw (days),ΔI (days),anchors,rows,entities,held-out entities,updates,first update,last update,load s
task,,,,,,,,,,,
driver-position,regression,60,1826,349,9518,846,81,14,1955-01-04,2020-01-01,3.6
driver-dnf,binary_classification,30,1826,588,15013,855,81,14,1955-01-04,2020-01-01,3.6
driver-top3,binary_classification,30,1826,228,4973,168,15,5,2000-01-02,2020-01-01,4.4


## What each window holds

Window 1 is the initial history; the model updated at T_e is scored on window e+1.
A window's held-out rows decide when the update that labels them stops.

In [6]:
def window_profile(ctx):
    r = ctx.rows[ctx.rows.window >= 1]
    g = r.groupby("window")
    prof = pd.DataFrame(
        {
            "start": g["time"].min().dt.date,
            "end": g["time"].max().dt.date,
            "anchors": g["time"].nunique(),
            "rows": g.size(),
            "entities": g["entity"].nunique(),
            "held-out rows": g["holdout"].sum(),
            "held-out entities": r[r.holdout].groupby("window")["entity"].nunique(),
        }
    ).fillna(0)
    if ctx.task.task_type.value == "binary_classification":
        prof["positive rate"] = g["target"].mean().round(3)
    else:
        prof["target mean"] = g["target"].mean().round(2)
    return prof


profiles = {t: window_profile(ctx) for t, ctx in contexts.items()}
for t, prof in profiles.items():
    print(f"\n{t}")
    display(prof)


driver-position


,start,end,anchors,rows,entities,held-out rows,held-out entities,target mean
window,,,,,,,,
1,1950-06-19,1954-11-25,18,660,211,51,14,13.64
2,1955-03-25,1959-12-29,19,663,195,73,19,12.98
3,1960-04-27,1964-12-02,21,667,171,77,21,14.12
4,1965-04-01,1969-09-07,23,494,84,21,5,10.31
5,1970-03-06,1974-12-10,24,721,105,36,7,13.92
6,1975-02-08,1979-11-14,28,882,112,97,14,15.07
7,1980-01-13,1984-10-18,23,706,81,79,12,15.13
8,1985-02-15,1989-09-22,23,713,70,120,11,15.84
9,1990-01-20,1994-10-26,26,834,84,115,9,16.03



driver-dnf


,start,end,anchors,rows,entities,held-out rows,held-out entities,positive rate
window,,,,,,,,
1,1950-05-20,1954-12-25,26,839,221,66,14,0.892
2,1955-04-24,1959-11-29,27,779,189,84,17,0.828
3,1960-01-28,1964-12-02,31,864,178,100,22,0.881
4,1965-05-01,1969-10-07,36,713,84,30,5,0.871
5,1970-03-06,1974-09-11,41,1124,105,53,7,0.877
6,1975-01-09,1979-12-14,46,1352,112,146,14,0.875
7,1980-01-13,1984-10-18,41,1203,81,130,12,0.914
8,1985-03-17,1989-10-22,41,1244,70,210,11,0.926
9,1990-02-19,1994-10-26,46,1412,84,192,9,0.910



driver-top3


,start,end,anchors,rows,entities,held-out rows,held-out entities,positive rate
window,,,,,,,,
1,1994-02-28,1999-04-03,35,853,69,118,6,0.151
2,2000-02-27,2004-10-03,24,500,43,51,4,0.204
3,2005-03-02,2009-10-07,42,891,47,114,4,0.213
4,2010-03-06,2014-11-10,48,1118,47,183,7,0.174
5,2015-03-10,2019-11-14,47,964,39,187,6,0.196
6,2020-06-11,2023-07-26,32,647,32,81,4,0.206


## Native RelBench tables vs. the uniform grid

RelBench's native tables use only `num_eval_timestamps` anchors from the validation
and from the test timestamp. Its test anchors start at `test_timestamp`, which is
generally not a whole number of Δw after the validation timestamp, so they sit off
the uniform grid. Counted here over anchors that produce at least one row.

In [7]:
gap_rows = []
for task_name, ctx in contexts.items():
    task, db = ctx.task, ctx.db
    native_with_rows, native_test = set(), set()
    for split in ("train", "val", "test"):
        tab = task.get_table(split, mask_input_cols=False).df
        ts = set(pd.to_datetime(tab[task.time_col].unique()))
        native_with_rows |= ts
        if split == "test":
            native_test = ts
    grid = set(pd.to_datetime(ctx.rows["time"].unique()))
    gap_rows.append(
        {
            "task": task_name,
            "num_eval_timestamps": task.num_eval_timestamps,
            "native anchors with rows": len(native_with_rows),
            "grid anchors with rows": len(grid),
            "on both": len(grid & native_with_rows),
            "grid only (gaps filled)": len(grid - native_with_rows),
            "native test anchors off the grid": len(native_test - grid),
        }
    )
pd.DataFrame(gap_rows).set_index("task")

,num_eval_timestamps,native anchors with rows,grid anchors with rows,on both,grid only (gaps filled),native test anchors off the grid
task,,,,,,
driver-position,40,310,349,277,72,33
driver-dnf,40,475,588,446,142,29
driver-top3,40,116,228,86,142,30


## Step-time benchmark

One update's training loop on the device this notebook runs on: driver-dnf at its
last update (largest training set), cumulative mixing over all labelled history.

In [8]:
from redelex.loaders import MixtureLoader, Source
from redelex.nn.train import LightningEntityTaskWrapper
from redelex.transforms import AttachTargetTransform
from experiments.continuous_learning.chain_runner import new_model, training_sources
from experiments.continuous_learning.modes import get_mode
from experiments.continuous_learning.utils import get_table_input

bench_task = "driver-dnf" if "driver-dnf" in contexts else TASKS[0]
ctx = contexts[bench_task]
cfg = ChainConfig(DATASET, bench_task, "cumulative", 0, OUT, device=DEVICE)
device = ctx.device
ep = ctx.plan.episodes[-1]
sources = training_sources(ctx, get_mode("cumulative"), ep)
srcs = {}
for name, (table, share) in sources.items():
    inp = get_table_input(table, ctx.task)
    srcs[name] = Source(inp.nodes[1], inp.time, inp.target, share)
steps = 30 if SCALE == "smoke" else 200
loader = MixtureLoader(ctx.data, ctx.task.entity_table, srcs, [32, 16], 128, steps * 128, seed=0,
                       transform_factory=AttachTargetTransform)
model = new_model(ctx, cfg).to(device)
wrapper = LightningEntityTaskWrapper(model, torch.optim.Adam(model.parameters(), lr=1e-3), ctx.task, modes=("val",))
opt = wrapper.optimizer
times, data_times = [], []
it = iter(loader)
for i in range(steps):
    t0 = time.perf_counter()
    batch = next(it).to(device)
    t1 = time.perf_counter()
    pred, target = wrapper(batch)
    loss = wrapper.loss_fn(pred.float(), target)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if device.type == "cuda":
        torch.cuda.synchronize()
    t2 = time.perf_counter()
    if i >= 3:  # skip warm-up
        data_times.append(t1 - t0)
        times.append(t2 - t1)

bench = {
    "task": bench_task,
    "device": env["device_name"],
    "threads": torch.get_num_threads(),
    "train_rows": int(sum(len(s.nodes) for s in srcs.values())),
    "steps_timed": len(times),
    "compute_s_per_step": float(np.median(times)),
    "data_s_per_step": float(np.median(data_times)),
    "s_per_step": float(np.median(np.add(times, data_times))),
}
bench["samples_per_s"] = 128 / bench["s_per_step"]
pd.Series(bench)

/home/pelesjak/redelex/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


task                  driver-dnf
device                    x86_64
threads                        8
train_rows                 12649
steps_timed                   27
compute_s_per_step      0.708829
data_s_per_step          0.01362
s_per_step              0.721932
samples_per_s         177.301969
dtype: object

In [9]:
record = {
    "env": env,
    "summary": summary.reset_index().astype(str).to_dict(orient="records"),
    "windows": {t: p.reset_index().astype(str).to_dict(orient="records") for t, p in profiles.items()},
    "benchmark": bench,
}
path = out / f"00_profile_{DATASET}_{'gpu' if device.type == 'cuda' else 'cpu'}.json"
path.write_text(json.dumps(record, indent=2))
print("saved", path)

saved notebooks/methodology/out/00_profile_rel-f1_cpu.json


## Gate

In [10]:
main_tasks = [t for t in TASKS if contexts[t].plan.num_updates >= 5]
appendix_tasks = [t for t in TASKS if t not in main_tasks]
print("main-grid tasks:", main_tasks)
print("appendix tasks (< 5 updates):", appendix_tasks)

for t, ctx in contexts.items():
    assert ctx.plan.num_updates >= 1 and ctx.plan.num_windows == ctx.plan.num_updates + 1, t
    prof = profiles[t]
    with_rows = prof[prof["rows"] > 0]
    assert (with_rows["held-out entities"] >= 1).all(), f"{t}: a window has rows but no held-out entity"
assert np.isfinite(bench["s_per_step"]) and bench["s_per_step"] > 0
assert path.exists()
print("GATE 00 PASSED")

main-grid tasks: ['driver-position', 'driver-dnf', 'driver-top3']
appendix tasks (< 5 updates): []
GATE 00 PASSED
